<a href="https://colab.research.google.com/github/seleneyong/ISYS2001-MyWork/blob/main/Copy_of_worksheet_designing_for_your_user.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevin-blasiak-curtin/ISYS2001-Archive/blob/main/Module%2010%20-%20Designing%20for%20your%20user/worksheet-designing-for-your-user.ipynb)

# Designing for your user

Last week you built your first Gradio app. This week the question changes from "does it run?" to "can the person I built it for actually use it?"

This worksheet has two parts:

- **Part A** runs the same small app twice. The first version works only when everything goes right. The second version is built around what real users actually do. All the code is written for you: run it, break it, and compare.
- **Part B** asks you to map out how a user moves through an app, step by step, and what could go wrong along the way.

Run the cells from top to bottom. If something stops working, use **Runtime > Restart session** and run everything again from the top.

---
## Setup

Gradio is not installed in Colab by default, so install it first. This takes around a minute.

In [1]:
!pip install -q gradio

In [2]:
import gradio as gr
import pandas as pd

print("Gradio version:", gr.__version__)

Gradio version: 6.27.0


### Create the test files

The next cell creates six small files in this Colab session. Each one stands in for something a real user might do.

| File | What it represents |
|------|--------------------|
| `sales_clean.csv` | Exactly what the app expects |
| `messy_data.csv` | Someone typed "FIVE" and "twelve" instead of numbers |
| `empty.csv` | An export with column headers but no rows |
| `wrong_columns.csv` | An export from a different system, with different column names |
| `wrong_type.txt` | The wrong file altogether |
| `notes_renamed.csv` | A text file someone renamed to `.csv` |

You do not need to download these. Each app shows them as clickable examples underneath the upload box.

In [3]:
test_files = {
    "sales_clean.csv": "product,quantity,price\nWidget A,10,25.50\nWidget B,5,40.00\nWidget C,15,12.75\n",
    "messy_data.csv": "product,quantity,price\nWidget A,10,25.50\nWidget B,FIVE,40.00\nWidget C,15,twelve\n",
    "empty.csv": "product,quantity,price\n",
    "wrong_columns.csv": "item,amount,cost\nWidget A,10,25.50\n",
    "wrong_type.txt": "This is not a CSV file!\nJust some random text.\nNothing to see here.\n",
    "notes_renamed.csv": "Meeting notes\nCall the supplier on Tuesday.\nAsk about the March invoice.\n",
}

for filename, contents in test_files.items():
    with open(filename, "w") as f:
        f.write(contents)
    print("Created", filename)

Created sales_clean.csv
Created messy_data.csv
Created empty.csv
Created wrong_columns.csv
Created wrong_type.txt
Created notes_renamed.csv


---
# Part A: One app, two versions

The app is deliberately simple. A sales manager uploads a CSV export with `product`, `quantity` and `price` columns, and the app reports total revenue. With `sales_clean.csv` the correct answer is **$646.25**.

## Version 1: the happy path only

This version works perfectly, as long as the user does exactly what the programmer expected.

**Before you run it, predict.** For each test file, write down what you expect to happen: a correct answer, a wrong answer, or a crash? Then run the cell, click each example under the upload box, and press **Submit**. Also try pressing **Submit** with no file at all.

In [ ]:
def calculate_revenue(file_path):
    df = pd.read_csv(file_path)
    df["revenue"] = df["quantity"] * df["price"]
    total = df["revenue"].sum()
    return f"Total revenue: ${total:,.2f}"


version_1 = gr.Interface(
    fn=calculate_revenue,
    inputs=gr.File(label="Upload file"),
    outputs=gr.Textbox(label="Result"),
    examples=[
        ["sales_clean.csv"],
        ["messy_data.csv"],
        ["empty.csv"],
        ["wrong_columns.csv"],
        ["wrong_type.txt"],
    ],
    title="Revenue app: version 1",
)

version_1.launch()

### What you should notice

- Most mistakes produce the same red **Error**. The user cannot tell whether the problem is their file, their data or the app, so they have no idea what to do next.
- `wrong_type.txt` does not fail because it is a text file. pandas happily reads almost anything as a table. It fails later, because there is no `quantity` column.
- `empty.csv` is the most dangerous one. It does not crash at all. It reports **$0.00**, which looks like a real answer. A manager could take that number into a meeting.

A crash is annoying, but a confident wrong answer is worse. Robustness is not only about stopping crashes. It is about making sure the user can trust what they see.

## Version 2: built around the user

The calculation is the same. What changed is that this version was planned around what a real user might do. It improves things in two ways:

- **The interface prevents some mistakes.** The upload box says what file it wants and only accepts `.csv` files, the button says what it will do, and the result confirms what happened.
- **The code catches the rest and explains them.** Checks and a `try...except` block turn each problem into a message that says what happened and what to do next.

Read the comments in the code before you run it. Each one explains *why* that part exists.

In [4]:
def process_sales(file_path):
    # Returns two things: a message for the result box, and a file to download
    # (or None when there is nothing to download).

    # The user clicked the button before uploading anything.
    if file_path is None:
        return "No file yet. Upload your monthly sales export, then click Calculate revenue.", None

    try:
        df = pd.read_csv(file_path)

        # Headers but no rows. Without this check the app reports $0.00.
        if df.empty:
            return "The file has column headers but no rows of data. Check that you exported the right month.", None

        # pd.to_numeric stops with a ValueError if a value is text, such as "twelve".
        df["quantity"] = pd.to_numeric(df["quantity"])
        df["price"] = pd.to_numeric(df["price"])
        df["revenue"] = df["quantity"] * df["price"]

        output_path = "processed_sales.csv"
        df.to_csv(output_path, index=False)

        total = df["revenue"].sum()
        message = f"Done. {len(df)} rows processed. Total revenue: ${total:,.2f}. Download the file below to see revenue for each product."
        return message, output_path

    # Specific problems first, the general catch-all last.
    except pd.errors.EmptyDataError:
        # Nothing in the file at all, not even headers.
        return "The file is completely empty. Check that the export finished, then upload it again.", None
    except pd.errors.ParserError:
        # The rows do not line up into a table.
        return "The file could not be read as a table. Check that every row has the same number of columns.", None
    except KeyError:
        # A required column is missing.
        return "The file needs columns called 'quantity' and 'price'. Check the headers in row 1, then upload it again.", None
    except ValueError:
        # Text where a number should be.
        return "Some quantity or price values are not numbers, for example 'FIVE' or 'twelve'. Fix those cells, then upload the file again.", None
    except Exception as e:
        # Anything we did not anticipate.
        return f"Something unexpected went wrong: {e}", None


with gr.Blocks(title="Sales revenue checker") as version_2:
    gr.Markdown("# Sales revenue checker")
    gr.Markdown("Upload last month's sales export to see total revenue, and download a copy with revenue for each product.")

    upload = gr.File(
        label="Monthly sales export (CSV with product, quantity and price columns)",
        file_types=[".csv"],
    )
    gr.Examples(
        examples=[["sales_clean.csv"], ["messy_data.csv"], ["empty.csv"], ["wrong_columns.csv"], ["notes_renamed.csv"]],
        inputs=upload,
        label="Example files to try",
    )

    run = gr.Button("Calculate revenue", variant="primary")
    result = gr.Textbox(label="Result", lines=2)
    download = gr.File(label="Processed file")

    run.click(process_sales, inputs=upload, outputs=[result, download])

version_2.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://125cd59bac8ddcc6dc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### What you should notice

- **Every mistake gets its own message**, written in the user's language: "column", "row", "cell". Not `KeyError` or `NoneType`.
- **`empty.csv` no longer produces a misleading $0.00.** The app says plainly that there is no data.
- **`wrong_type.txt` is gone from the examples**, because the upload box only accepts `.csv` files. If you upload a `.txt` file from your own computer, Gradio rejects it before the function even runs.
- **Prevention is not a guarantee.** `notes_renamed.csv` has the right ending but the wrong contents. It gets through the upload box, and the `except KeyError` block catches it. That is why version 2 uses both.
- **Many of the improvements are words, not code.** The label, the button text and the result message needed no new Python skills. They needed someone to think about the user.

**Try this:** open `processed_sales.csv` from the download box. Does it contain what the result message promised?

### Where the difference really came from

Version 2 did not come from a cleverer prompt like "add error handling" or "make this more robust". Prompts like that get generic results. It came from first listing what a real user might do: click before uploading, export the wrong month, type "FIVE" into a spreadsheet, rename a file. Whether you write the code yourself or ask AI to help, that list is what makes the result good. Part B is about building that list for an app of your own.

---
# Part B: Map out your user's path

Think of an app you would like to build, or use the app you are building for your project. Work through the steps below in order.

In a full software project, this kind of planning usually happens before any code is written, and the results are kept with the project as part of its documentation. If your app already exists, that is fine. Doing this now still shows you what to fix, and it gives you a record of *why* your app works the way it does. Keep what you produce somewhere you can find it again.

Double-click any text cell below to edit it.

## B1. Start with a user story

Teams often describe what an app needs to do in a short sentence called a **user story**. It has three parts: who the user is, what they want to do, and why. For the sales app in Part A:

> As a **regional sales manager**, I want to **check last month's total revenue**, so that **I have the figure ready for my Monday meeting**.

Write a user story for the main thing someone does in your app. Be specific about the user. "A café owner who checks stock levels on a Friday afternoon" is far more useful than "a business user".

*My app, in one sentence:*

*My user story:* As a ..., I want to ..., so that ...

## B2. Map the journey step by step

Now write down every step the user takes to complete that story, from opening the app to finishing the task. Write it like pseudocode: one action per line, in order, using the actual words on the screen where you can.

Here is the journey for the sales app:

```
1. Open the app
2. Read the title and description to check this is the right tool
3. Click the upload box
4. Choose last month's export from the CRM
5. Click "Calculate revenue"
6. Read the total in the result box
7. Click the processed file to download revenue for each product
```

Write yours in the cell below. If you find a step that feels awkward to write down, such as "work out which button to press", that is usually a sign the app could be clearer at that point.

```
1. Open the app
2.
3.
4.
5.
```

## B3. Add what could go wrong

Go back through your journey and, at each step, ask: what might a real user do here that I did not plan for? Add those as branches, the same way you would write an `if` in code. For each branch, write what the app should say and where the user goes next.

Here are some of the branches for the sales app:

```
3. Click the upload box
   IF they choose a file that is not a CSV
      -> the upload box does not accept it
4. Choose last month's export from the CRM
5. Click "Calculate revenue"
   IF no file was uploaded
      -> "No file yet. Upload your monthly sales export, then click Calculate revenue."
      -> back to step 3
   IF the file has no quantity or price column
      -> "The file needs columns called 'quantity' and 'price'. Check the headers in row 1..."
      -> fix the file, back to step 3
   IF the file has headers but no rows
      -> "The file has column headers but no rows of data..."
      -> export again, back to step 3
6. Read the total in the result box
```

Aim for at least three branches in your own journey. Some questions to get you started: What if they click a button before entering anything? What if they type text where you expect a number? What if a file, a search or an API returns nothing? What if they enter something valid but unusual, such as a negative amount or a date in the future?

Copy your journey from B2 into the cell below and add your branches.

```
1. Open the app
2.
   IF
      ->
      ->
3.
```

## B4. Organise the information

The way an app groups and labels its information is called its **information architecture**. For a small app it comes down to a few questions: what information does the user give the app, what does the app show back, how is it grouped, and what is each part called?

If your app has more than one screen, tab or section, list each one. For each, note what it is for in one sentence, what information appears there, and how the user gets to it. If you cannot describe a section's purpose in one sentence, it may be trying to do too much, or it may not need to exist.

For the sales app this is a single screen: one input (the export file), two outputs (the result message and the processed file), and labels that use the manager's words, not the programmer's.

| Screen, tab or section | What it is for | What information appears | How the user gets there |
|------------------------|----------------|--------------------------|-------------------------|
| | | | |
| | | | |

*Labels I will use, and why they make sense to my user:*

## B5. Sketch it

Finally, draw the main screen as a **wireframe**: boxes and labels, no colour, no styling. A box with a cross through it stands for an image, and wavy lines stand for text. Number the boxes to match the steps in your journey, so you can see the path on the page.

Choose whichever way of sketching is quickest for you:

- **Pen and paper.** This is genuinely the fastest option. Photograph the sketch with your phone when you are done.
- **[Excalidraw](https://excalidraw.com)** works in the browser with no account, and has a hand-drawn look that keeps you from fussing over details.
- **[diagrams.net](https://app.diagrams.net)** (also called draw.io) is free and has ready-made shapes for buttons, text boxes and screens.
- **[Penpot](https://penpot.app)** is a free, open-source design tool. It needs an account, and it is worth a look if you want to go further.

If your app has several screens, you can also draw them side by side with arrows between them, following your journey from B2.

When your sketch is ready, add it to the text cell below. In Colab, double-click the cell and use the image button in the text cell toolbar, or drag the image file into the cell. Keep the original file with the rest of your project as well.

*My sketch:*

---
## Looking ahead

Next week is about testing. Hold on to your journey from B3: every `IF` branch in it is a test case waiting to be run.